# Notebook 4: The Oklahoma Experiment

Tracking the full statistical lifecycle of induced seismicity in Oklahoma.

In [1]:
import matplotlib
matplotlib.use('Agg')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import estimate_b_value, bootstrap_b_value, rolling_b_value
from src.interevent import compute_interevent_times, fit_distributions, classify_regime
from src.external_data import load_occ_uic_volumes, aggregate_oklahoma_injection
from src.plotting import setup_style, save_figure, plot_oklahoma_timeline

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_oklahoma.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} Oklahoma events")
print(f"Date range: {df['time'].min()} → {df['time'].max()}")
print(f"Magnitude range: {df['mag'].min():.1f} → {df['mag'].max():.1f}")

Loaded 31,187 Oklahoma events
Date range: 2000-10-08 10:16:23.780000+00:00 → 2025-12-31 13:16:31.969000+00:00
Magnitude range: 1.0 → 5.8


## 4.1a Injection Volume Data

We load monthly wastewater injection volumes from the Oklahoma Corporation Commission (OCC) UIC well data (2006–2024). This is the key independent variable — the link between industrial activity and seismicity.

In [3]:
# Load OCC UIC injection volumes (2006-2024, monthly per-well)
uic_df = load_occ_uic_volumes(years=list(range(2006, 2025)))
print(f"Total UIC records: {len(uic_df):,}")

# Aggregate to monthly Oklahoma totals
injection_monthly = aggregate_oklahoma_injection(uic_df)
print(f"\nMonthly injection volume summary ({injection_monthly['date'].min().year}-{injection_monthly['date'].max().year}):")
print(f"  Peak month: {injection_monthly.loc[injection_monthly['total_volume_bbl'].idxmax(), 'date'].strftime('%Y-%m')} "
      f"({injection_monthly['total_volume_bbl'].max()/1e6:.0f}M bbl)")
print(f"  Last month: {injection_monthly.iloc[-1]['date'].strftime('%Y-%m')} "
      f"({injection_monthly.iloc[-1]['total_volume_bbl']/1e6:.0f}M bbl)")
print(f"  Peak active wells: {injection_monthly['n_wells'].max():,}")

Total UIC records: 1,935,853



Monthly injection volume summary (2006-2024):
  Peak month: 2010-06 (1800M bbl)
  Last month: 2024-12 (106M bbl)
  Peak active wells: 8,253


## 4.1 Phase Definition

In [4]:
phases = {
    "Baseline": ("2000-01-01", "2009-01-01"),
    "Onset": ("2009-01-01", "2013-01-01"),
    "Surge": ("2013-01-01", "2016-07-01"),
    "Regulation": ("2016-07-01", "2020-01-01"),
    "Recovery": ("2020-01-01", "2026-01-01"),
}

phase_colors = {
    "Baseline": "#457B9D",
    "Onset": "#2A9D8F",
    "Surge": "#E63946",
    "Regulation": "#F4A261",
    "Recovery": "#6A4C93",
}

# Create phase boundaries as tz-aware Timestamps
phase_starts = {name: pd.Timestamp(start, tz="UTC") for name, (start, end) in phases.items()}
phase_ends = {name: pd.Timestamp(end, tz="UTC") for name, (start, end) in phases.items()}

def assign_phase(t):
    for name in phases:
        if phase_starts[name] <= t < phase_ends[name]:
            return name
    return "Unknown"

df["phase"] = df["time"].apply(assign_phase)
print(df["phase"].value_counts())

phase
Recovery      14767
Surge          9923
Regulation     5965
Onset           488
Baseline         44
Name: count, dtype: int64


## 4.2 Seismicity Timeline

In [5]:
# Monthly event counts (M2.5+)
mc_global = estimate_mc(df["mag"].values)
df_above = df[df["mag"] >= max(mc_global, 2.5)].copy()
df_above["month"] = df_above["time"].dt.to_period("M").dt.to_timestamp()
monthly = df_above.groupby("month").size().reset_index(name="count")

# Rolling b-value
rolling = rolling_b_value(df_above["time"].values, df_above["mag"].values,
                          mc=max(mc_global, 2.5), window_size=100, step=20)
rolling["center_time"] = pd.to_datetime(rolling["center_time"])

# Phase boundaries
phase_boundaries = [(pd.Timestamp(start, tz="UTC"), name) 
                    for name, (start, _) in phases.items()
                    if name != "Baseline"]

fig, ax = plt.subplots(figsize=(14, 7))
plot_oklahoma_timeline(monthly, rolling, phase_boundaries, ax=ax)
save_figure(fig, "04_oklahoma_timeline")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/1091490013.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_above["month"] = df_above["time"].dt.to_period("M").dt.to_timestamp()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/1091490013.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.2a Seismicity vs. Injection Volume

This is the central figure: monthly earthquake counts overlaid with total monthly wastewater injection volumes from OCC records. The dual-axis plot reveals the temporal relationship between injection activity and seismicity.

In [6]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Seismicity (left axis)
ax1.bar(monthly["month"], monthly["count"], width=25, color="#E63946", alpha=0.6, label="Earthquakes (M2.5+)")
ax1.set_xlabel("Date")
ax1.set_ylabel("Monthly Earthquake Count", color="#E63946")
ax1.tick_params(axis="y", labelcolor="#E63946")

# Injection volume (right axis)
ax2 = ax1.twinx()
ax2.plot(injection_monthly["date"], injection_monthly["total_volume_bbl"] / 1e6,
         color="#457B9D", linewidth=2, label="Injection Volume")
ax2.fill_between(injection_monthly["date"], 0, injection_monthly["total_volume_bbl"] / 1e6,
                 alpha=0.15, color="#457B9D")
ax2.set_ylabel("Monthly Injection Volume (million bbl)", color="#457B9D")
ax2.tick_params(axis="y", labelcolor="#457B9D")

# Phase boundaries
for name, (start, _) in phases.items():
    if name != "Baseline":
        ax1.axvline(pd.Timestamp(start), color="black", linestyle=":", alpha=0.4)
        ax1.text(pd.Timestamp(start), ax1.get_ylim()[1] * 0.95, f" {name}", fontsize=9, ha="left")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Oklahoma Seismicity vs. Wastewater Injection Volume", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "04_injection_vs_seismicity")
plt.show()

# Compute lag correlation
from scipy.stats import pearsonr
# Merge on month
monthly_merged = monthly.copy()
monthly_merged["month_dt"] = pd.to_datetime(monthly_merged["month"])
inj = injection_monthly[["date", "total_volume_bbl"]].copy()
inj["month_dt"] = inj["date"].dt.to_period("M").dt.to_timestamp()

merged = monthly_merged.merge(inj, on="month_dt", how="inner")
if len(merged) > 10:
    # Test lags 0 to 12 months
    best_r, best_lag = 0, 0
    for lag in range(0, 13):
        if lag == 0:
            r, p = pearsonr(merged["total_volume_bbl"], merged["count"])
        else:
            r, p = pearsonr(merged["total_volume_bbl"].iloc[:-lag], merged["count"].iloc[lag:])
        if abs(r) > abs(best_r):
            best_r, best_lag = r, lag
    print(f"\nPeak correlation: r={best_r:.3f} at lag={best_lag} months")
    r0, p0 = pearsonr(merged["total_volume_bbl"], merged["count"])
    print(f"Zero-lag correlation: r={r0:.3f}, p={p0:.2e}")


Peak correlation: r=0.032 at lag=11 months
Zero-lag correlation: r=-0.008, p=9.15e-01


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/551703942.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.3 Spatial Evolution by Phase

In [7]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)

for i, (phase_name, (start, end)) in enumerate(phases.items()):
    ax = axes[i]
    mask = df["phase"] == phase_name
    phase_df = df[mask]
    
    sc = ax.scatter(phase_df["longitude"], phase_df["latitude"],
                    c=phase_df["depth"], cmap="viridis_r", s=2, alpha=0.3,
                    vmin=0, vmax=15)
    ax.set_xlim(-100, -94.5)
    ax.set_ylim(33.5, 37.5)
    ax.set_title(f"{phase_name}\n({len(phase_df):,} events)", fontsize=10)
    ax.set_xlabel("Longitude (\u00b0)")
    if i == 0:
        ax.set_ylabel("Latitude (\u00b0)")

plt.colorbar(sc, ax=axes, label="Depth (km)", shrink=0.8)
fig.suptitle("Oklahoma Seismicity by Phase \u2014 Color: Depth", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "04_spatial_evolution")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/2400953456.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/2400953456.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.4 Multi-Metric Phase Comparison

In [8]:
phase_metrics = {}

for phase_name, (start, end) in phases.items():
    mask = df["phase"] == phase_name
    phase_df = df[mask].copy()
    
    if len(phase_df) == 0:
        continue
    
    mc = estimate_mc(phase_df["mag"].values)
    above_mc = phase_df[phase_df["mag"] >= mc]
    
    # Monthly event count
    phase_df["month"] = phase_df["time"].dt.to_period("M")
    monthly_counts = phase_df.groupby("month").size()
    
    # b-value with bootstrap CI
    if len(above_mc) >= 30:
        b_mean, b_lo, b_hi = bootstrap_b_value(above_mc["mag"].values, mc, n_bootstrap=1000)
    else:
        b_mean, b_lo, b_hi = np.nan, np.nan, np.nan
    
    # Weibull k
    if len(above_mc) >= 50:
        times_sorted = above_mc.sort_values("time")["time"].values
        iet = compute_interevent_times(times_sorted)
        iet_hours = iet / 3600.0
        iet_hours = iet_hours[iet_hours > 0]
        try:
            fit = fit_distributions(iet_hours)
            weibull_k = fit["weibull"]["params"][0]
        except Exception:
            weibull_k = np.nan
    else:
        weibull_k = np.nan
    
    # Spatial centroid
    centroid_lat = phase_df["latitude"].mean()
    centroid_lon = phase_df["longitude"].mean()
    
    # Max magnitude per quarter
    phase_df_q = phase_df.copy()
    phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
    max_mag_quarter = phase_df_q.groupby("quarter")["mag"].max()
    
    phase_metrics[phase_name] = {
        "n_events": len(phase_df),
        "monthly_mean": monthly_counts.mean(),
        "monthly_std": monthly_counts.std(),
        "mc": mc,
        "b_value": b_mean,
        "b_ci_low": b_lo,
        "b_ci_high": b_hi,
        "weibull_k": weibull_k,
        "centroid_lat": centroid_lat,
        "centroid_lon": centroid_lon,
        "max_mag_mean": max_mag_quarter.mean(),
    }

metrics_df = pd.DataFrame(phase_metrics).T
print(metrics_df.to_string())

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_4

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")


            n_events  monthly_mean  monthly_std   mc   b_value  b_ci_low  b_ci_high  weibull_k  centroid_lat  centroid_lon  max_mag_mean
Baseline        44.0      1.466667     0.819307  3.1       NaN       NaN        NaN        NaN     34.881932    -97.164614      2.971429
Onset          488.0     10.382979    11.435173  2.7  0.983586  0.885102   1.094954   0.542718     35.455395    -97.062904      3.700000
Surge         9923.0    236.261905   171.042722  2.7  1.162881  1.133026   1.190517   0.434857     36.548039    -97.724434      4.335714
Regulation    5965.0    142.023810    74.540704  2.7  1.197347  1.148332   1.247794   0.650585     36.317507    -97.727655      4.275714
Recovery     14767.0    205.097222   113.231429  1.4  0.934855  0.917654   0.951900   0.685214     35.711263    -97.537702      3.592917


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")


## 4.5 Weibull k Evolution

In [9]:
# Compute Weibull k in rolling yearly windows
mc_ok = estimate_mc(df["mag"].values)
df_above_mc = df[df["mag"] >= mc_ok].sort_values("time")

k_values = []
years = range(2001, 2025)
for year in years:
    t_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    t_end = pd.Timestamp(f"{year+1}-01-01", tz="UTC")
    mask = (df_above_mc["time"] >= t_start) & (df_above_mc["time"] < t_end)
    year_events = df_above_mc[mask]
    
    if len(year_events) < 30:
        k_values.append(np.nan)
        continue
    
    iet = compute_interevent_times(year_events["time"].values)
    iet_hours = iet / 3600.0
    iet_hours = iet_hours[iet_hours > 0]
    
    if len(iet_hours) < 20:
        k_values.append(np.nan)
        continue
    
    try:
        fit = fit_distributions(iet_hours)
        k_values.append(fit["weibull"]["params"][0])
    except Exception:
        k_values.append(np.nan)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(list(years), k_values, "o-", color="#E63946", linewidth=1.5, markersize=5)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="k=1 (Poisson)")

# Phase boundaries
for name, (start, _) in phases.items():
    if name != "Baseline":
        ax.axvline(int(start[:4]), color="black", linestyle=":", alpha=0.4)
        ax.text(int(start[:4]), ax.get_ylim()[1]*0.95, f" {name}", fontsize=8, ha="left")

ax.set_xlabel("Year")
ax.set_ylabel("Weibull shape parameter k")
ax.set_title("Oklahoma IET Weibull k Evolution")
ax.legend()
save_figure(fig, "04_weibull_k_evolution")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/1138603733.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.6 Recovery Assessment

In [10]:
if "Baseline" in phase_metrics and "Recovery" in phase_metrics:
    baseline = phase_metrics["Baseline"]
    recovery = phase_metrics["Recovery"]
    
    metrics_compare = ["monthly_mean", "b_value", "mc", "weibull_k", "max_mag_mean"]
    labels = ["Monthly events", "b-value", "Mc", "Weibull k", "Max mag/quarter"]
    
    fig, axes = plt.subplots(1, len(metrics_compare), figsize=(15, 5))
    
    for i, (metric, label) in enumerate(zip(metrics_compare, labels)):
        ax = axes[i]
        vals = [baseline.get(metric, np.nan), recovery.get(metric, np.nan)]
        colors = [phase_colors["Baseline"], phase_colors["Recovery"]]
        bars = ax.bar(["Baseline", "Recovery"], vals, color=colors, alpha=0.8)
        
        if metric == "b_value":
            # Add CI whiskers
            for j, phase in enumerate(["Baseline", "Recovery"]):
                pm = phase_metrics[phase]
                ax.errorbar(j, pm["b_value"], 
                           yerr=[[pm["b_value"]-pm["b_ci_low"]], [pm["b_ci_high"]-pm["b_value"]]],
                           fmt="none", color="black", capsize=5)
        
        ax.set_title(label, fontsize=10)
        ax.set_ylabel(label)
    
    fig.suptitle("Oklahoma: Baseline vs. Recovery Phase Comparison", fontsize=13, fontweight="bold")
    fig.tight_layout()
    save_figure(fig, "04_recovery_assessment")
    plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_40898/1046647117.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook tracked Oklahoma's induced seismicity through five distinct phases:

1. **Baseline (2000-2008):** Low background seismicity with typical tectonic b-values and Poisson-like inter-event timing.
2. **Onset (2009-2012):** Seismicity begins climbing as wastewater injection volumes increase. Spatial patterns shift toward injection well clusters.
3. **Surge (2013-2016):** Dramatic increase in event rates, depressed b-values indicating stress-driven seismicity, and clustered (non-Poisson) inter-event times.
4. **Regulation (2016-2019):** Following volume reduction mandates, seismicity rates decline but statistical signatures of induced activity persist.
5. **Recovery (2020-2025):** Rates approach baseline levels, but key metrics (b-value, Weibull k, spatial footprint) reveal whether the crust has fully returned to its pre-injection state.

The multi-metric comparison between Baseline and Recovery phases quantifies the degree to which Oklahoma's seismicity has (or has not) returned to a natural tectonic regime.